# Cluster Overview

This notebook queries the OCP audit SQLite datastore to present the **Cluster Overview** report — a single-row identity card per cluster covering:

- Version & Identity
- Platform & Topology
- Node Sizing
- Network Configuration
- Access Endpoints
- Update Posture

In [ ]:
import os
import sys

import pandas as pd

# Shared notebook helpers (sys.path + OCP_AUDIT_DB + styling)
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from notebook_style import bootstrap, style_table  # noqa: E402

session, engine = bootstrap()

from schema.models import Cluster, ClusterOverview  # noqa: E402

print(f"Connected to: {engine.url}")


## Cluster Inventory

In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

---
## Version & Identity

OCP version, Kubernetes version, cluster ID, install date, and cluster age.

In [ ]:
df_version = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_version)

---
## Platform & Topology

Infrastructure platform and control-plane / infrastructure topology.

In [ ]:
df_platform = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_platform)

---
## Node Sizing

Master, worker, infra, and total node counts per cluster.

In [ ]:
df_nodes = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_nodes)

---
## Network Configuration

SDN type, cluster CIDRs, and service CIDRs.

In [ ]:
df_network = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_network)

---
## Access Endpoints

Console URL, API server URL, and default ingress domain.

In [ ]:
df_endpoints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.default_ingress_domain,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_endpoints)

---
## Update Posture

Current OCP version, update channel, update state, and count of available updates.

In [ ]:
df_updates = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.update_channel,
        ClusterOverview.update_state,
        ClusterOverview.available_updates_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_updates)

---
## Full Cluster Overview

All 21 fields for every cluster in a single table.

In [ ]:
df_full = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
        ClusterOverview.default_ingress_domain,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.update_channel,
        ClusterOverview.available_updates_count,
        ClusterOverview.update_state,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_full)} cluster overview(s)")
style_table(df_full)

In [ ]:
session.close()
print("Session closed.")